# SPARC Example 07: Rmax vs Vrot Parameter Space

**EPS Research RAG Astrophysics Corpus — Unified HI Corpus v7.3**

The Rmax vs Vrot_max diagram shows the full dynamic range:
from ~3 kpc dwarf irregulars to ~100 kpc massive spirals.
Colored by quality tier (Tier 1 = hand-curated, Tier 2 = automated).

**Important note on corpus fidelity:** The `rotation_curve_corpus_v7.1_flat.csv` and `rotation_curve_corpus_v7.json` are **full-fidelity** — not a summary or veneer. The CSV contains every kinematic parameter published by Lelli et al. (2016) including per-galaxy inclination, distance uncertainties, mass-to-light ratios, and rotation curve statistics. The JSON adds full per-ring data: Vobs, Vgas, Vdisk, Vbul, errV at every radial point. This is the complete published dataset in a single machine-readable file.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.23442703  
**Source:** Lelli, McGaugh & Schombert (2016), AJ 152, 157  
**Dependencies:** Python 3, numpy, matplotlib, csv (standard library only)

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/23442703/files/rotation_curve_corpus_v7.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import csv
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

rows = []
with open('rotation_curve_corpus_v7.1_flat.csv') as f:
    for r in csv.DictReader(f):
        try:
            rows.append({
                'survey': r['survey'],
                'r_max': float(r['r_max_kpc']),
                'v_max': float(r['vrot_max_kms']),
                'tier':  int(r['quality_tier']),
            })
        except (ValueError, KeyError):
            pass

SURVEYS = ['SPARC', 'THINGS', 'LITTLE_THINGS', 'WALLABY']
COLORS  = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
MARKERS = ['o', 's', '^', 'D']

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Panel 1: Rmax vs Vmax coloured by survey
ax = axes[0]
for s, c, m in zip(SURVEYS, COLORS, MARKERS):
    sub = [r for r in rows if r['survey']==s]
    ax.scatter([r['r_max'] for r in sub], [r['v_max'] for r in sub],
               s=15, alpha=0.5, color=c, marker=m, label=f'{s} (N={len(sub)})')
ax.set_xlabel('R_max (kpc)', fontsize=12)
ax.set_ylabel('Vmax (km/s)', fontsize=12)
ax.set_title('Parameter Space: R_max vs Vmax\nAll surveys', fontsize=11)
ax.legend(fontsize=8)

# Panel 2: Tier-1 only with log axes
ax2 = axes[1]
t1 = [r for r in rows if r['tier']==1]
for s, c, m in zip(SURVEYS, COLORS, MARKERS):
    sub = [r for r in t1 if r['survey']==s]
    if sub:
        ax2.scatter([r['r_max'] for r in sub], [r['v_max'] for r in sub],
                    s=20, alpha=0.6, color=c, marker=m, label=s)
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('R_max (kpc) [log]', fontsize=12)
ax2.set_ylabel('Vmax (km/s) [log]', fontsize=12)
ax2.set_title('Tier-1 Only — Log Scale', fontsize=11)
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3, which='both')

plt.suptitle('HI Corpus v7.3 — Rotation Curve Parameter Space', fontsize=11)
plt.tight_layout()
plt.savefig('ex07_rmax_vmax_parameter_space.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Total galaxies: {len(rows)}  Tier-1: {len(t1)}')
print(f'R_max range: {min(r["r_max"] for r in rows):.1f} — '
      f'{max(r["r_max"] for r in rows):.1f} kpc')
print(f'Vmax range:  {min(r["v_max"] for r in rows):.1f} — '
      f'{max(r["v_max"] for r in rows):.1f} km/s')
